# March Mania · Quality wins and schedule-adjusted records

**Feature investigation 04 · diagnose → reuse → engineer → test → checkpoint**

The 2019 shooting experiment worsened Brier for both populations. Notebook 03's automatic expansion is **paused**, not required. This notebook preserves the negative result, explains where loss accumulated using saved predictions, and tests **seven different features** on **2018**.

The classifier and the 16 reference inputs stay fixed. No new rating models are trained; 14 cached team-season snapshots are verified and reused. This milestone runs at most **eight new tournament classifiers**, not a feature-bank rebuild. These are already-used historical seasons, not untouched validation or the 2026 leaderboard.

Run the terminal tests in **START_HERE.md** first. Choose **Python (March Mania)** and restart the kernel before running all cells. No GPU, environment reinstall, cloud API calls, Git writes, or Kaggle submissions are involved.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown, FileLink

KIT = Path.cwd().resolve()
if not (KIT / 'run_round04.py').is_file():
    KIT = Path.home() / 'march_schedule_research'
REPO = Path(os.environ.get('MARCH_REPO', str(Path.home() / 'march-machine-learning-mania-2026'))).expanduser().resolve()
PRIOR = Path(os.environ.get('MARCH_SHOOTING_KIT', str(Path.home() / 'march_shooting_research'))).expanduser().resolve()
assert (KIT / 'run_round04.py').is_file(), 'Open the notebook inside march_schedule_research.'
assert (PRIOR / 'private_runs').is_dir(), 'Prior cache missing: do not rebuild or redownload automatically.'
sys.path.insert(0, str(KIT))
from run_round04 import run_stage
from schedule_plots import figures
pio.renderers.default = 'plotly_mimetype'
print('Kernel:', sys.executable)
print('Repository (read-only):', REPO)
print('Prior snapshots and model evidence (read-only):', PRIOR)
print('New private outputs:', KIT / 'private_runs')

## 1. Preserve the measured negative result

Positive `delta_vs_anchor` means a **worse** Brier score. Every shooting addition worsened both populations in 2019. The old `clearly_unproductive_smoke: false` flag required *all six* deltas to exceed 0.01; it was not evidence of improvement. Do not use it as an instruction to run notebook 03.

A single season cannot prove a feature is useless everywhere. Here the decision is to **pause promotion and large expansion**, preserve the evidence, and investigate a different, domain-supported representation.

In [ ]:
prior_metrics = pd.read_csv(KIT / 'evidence' / 'smoke_metrics.csv')
display(prior_metrics[['Gender','Season','recipe','brier','delta_vs_anchor']].assign(brier=lambda table: table['brier'].map('{:.7f}'.format), delta_vs_anchor=lambda table: table['delta_vs_anchor'].map('{:+.7f}'.format)))
print('These are your uploaded 2019 results, not newly fitted results.')

## 2. Reuse snapshots, replay saved predictions, and build seven features

**Family A · seed-weighted quality wins (3).** Reward wins over stronger seeded opponents, distinguish away/neutral achievements, and record losses to non-field opponents. This is a limited adaptation of a leading solution's concept, not a reproduction of its complete approach. Secondary tournament participation is excluded because its exact as-of availability has not been established.

**Family B · reference-team record (4).** Compare the observed record with a fixed 75th-percentile strength reference against the same opponents and venues. Separate total surplus, nonhome surplus, nonlinear difficult-win credit and costly-loss burden. These are custom proxies, **not NCAA NET or official Wins Above Bubble**.

Features use regular-season games through day 132 and seeds known after the field announcement. The prediction timestamp is after that announcement, not before Selection Sunday. Rating estimates may describe the whole pre-tournament season; they are not claimed to be pre-game ratings for each regular-season game. Definitions, heuristic constants, source links and limits appear in **RESEARCH_PLAN.md**.

Preparation also verifies eight cached 2019 prediction streams against their saved Brier scores and creates per-game loss diagnostics. It performs **zero new classifier or rating fits**. Hard stage limit: **300 seconds**; heartbeats: **15 seconds**.

In [ ]:
run_stage('prepare', max_seconds=300)
run_info = json.loads((KIT / 'reports' / 'latest_run.json').read_text())
RUN = Path(run_info['run_dir'])
print(json.dumps(json.loads((RUN / 'prepare.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'prior_replay.csv'))
display(pd.read_csv(RUN / 'feature_registry.csv').query('new_candidate == True')[['feature','family','description']])

### Where did the shooting features add loss?

The following table shows the largest positive paired loss changes from saved 2019 predictions. It identifies *where* deterioration occurred, not why a feature caused it. Team IDs are retained to avoid guessing team names. The full per-game table remains private; the return archive contains aggregate diagnostics only.

In [ ]:
prior_games = pd.read_csv(RUN / 'prior_game_diagnostics.csv')
display(prior_games.sort_values('paired_loss_delta', ascending=False).head(15)[
    ['Gender','recipe','Team1ID','Team2ID','y','anchor_probability','probability','paired_loss_delta']])
print('No 2019 model was refitted for this table. Do not turn individual outcomes into override rules.')

## 3. One controlled 2018 experiment

| Recipe | Fixed inputs | New family |
|---|---:|---|
| Anchor | 16 | None |
| Anchor + quality | 19 | Quality wins |
| Anchor + record | 20 | Reference-team record |
| Anchor + both | 23 | Both |

Men and women are fitted separately: **4 × 2 = 8 classifiers**. Training tournament years are **2013–2017**, validation is **2018**, main-draw games only. Logistic C=0.1, train-only RMS scaling, mirrored orientations and per-physical-game weighting remain identical to round 02. Only training-constant columns may be removed. No feature search, reweighting, calibration sweep or ensemble is performed.

2018 was selected as the next exploratory smoke season before observing these new-feature results. It is not an untouched test. Validation errors cannot be used to change this run's formulas. Hard evaluation limit: **180 seconds**.

In [ ]:
run_stage('evaluate', max_seconds=180)
metrics = pd.read_csv(RUN / 'metrics.csv')
display(metrics[['Gender','Season','recipe','games','train_games','brier','delta_vs_anchor','active_features']].assign(brier=lambda table: table['brier'].map('{:.7f}'.format), delta_vs_anchor=lambda table: table['delta_vs_anchor'].map('{:+.7f}'.format)))
display(pd.read_csv(RUN / 'ablations.csv'))

## 4. Check the decision, not just the smallest number

A delta at or below **−0.001** is a predeclared trigger to **consider an unchanged replication**, not to promote a feature. Smaller gains are inconclusive. Negative in both the add-alone and add-to-other comparisons is more useful evidence than a gain only in one configuration. Men and women may require different conclusions.

If every addition is worse, stop this direction as implemented and document it. If a feature looks promising, the next report should assess multiple season folds and stronger-model transfer before any submission claim. A technical `COMPLETE` does not mean the hypothesis succeeded.

In [ ]:
summary = json.loads((RUN / 'summary.json').read_text())
print(json.dumps(summary, indent=2))
display(pd.read_csv(RUN / 'training_redundancy.csv'))

## 5. Interactive evidence

These ten Plotly figures cover the prior negative result, error concentration, candidate families, team profiles, new Brier deltas, controlled family effects, calibration, and training-only correlation with the fixed anchor. Sparse calibration bins and single-season coefficients are descriptive, not causal evidence.

In [ ]:
plots = figures(RUN)
assert len(plots) == 10
for fig in plots[:5]:
    fig.show()

In [ ]:
for fig in plots[5:]:
    fig.show()

## 6. Save the checkpoint and return one small archive

Report rendering has a **120-second** ceiling. The archive is published only after repository, raw-input and upstream-cache preservation checks pass. It excludes raw game rows, fitted models, per-game predictions and private notebooks.

Save this notebook with **Ctrl+S**. Download `reports/milestone_04_return.zip` and attach it in ChatGPT. Stop this milestone here. Do not run the old shooting season panel, generate a submission or alter the canonical notebooks.

In [ ]:
run_stage('report', max_seconds=120)
record = json.loads((KIT / 'reports' / 'latest_report.json').read_text())
print('Return archive:', record['return_zip'])
print('Interactive HTML:', record['html'])
display(FileLink(str(Path(record['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(record['html']).relative_to(KIT))))
print('Completed work is retained under the fingerprinted private_runs directory.')